# 03 — Prime Density vs x/log(x)

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Purpose:** measure global prime-counting structure against the logarithmic density baseline.

Notebook 01 measured finite residue constraint.  
Notebook 02 measured local gap structure.  
Notebook 03 measures global counting structure.

\[
\pi(x) \approx \frac{x}{\log x}
\]

Here, \(\pi(x)\) counts primes less than or equal to \(x\).

## 0. Setup

Artifact structure:

```text
03_density_vs_log/
├── data/
├── docs/
├── figures/
└── tex/
```

Root export:

```text
03_density_vs_log_export.zip
```

In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "03_density_vs_log"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]
NOTEBOOK_TITLE = "Prime Density vs x/log(x)"

OUT = Path(NOTEBOOK_ID)
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
FIG_DIR = OUT / "figures"
TEX_DIR = OUT / "tex"

for d in [DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Artifact directory: {OUT.resolve()}")

## 1. Premise

Prime density measures global counting structure.

Terms:

- **continues:** prime counts increase across scale.
- **remains under constraint / persists:** prime counts remain near a logarithmic density law.
- **drift:** measurable deviation from the density baseline.
- **recoverability:** ability to recover global count scale from a simpler analytic baseline.

This notebook does not prove the Prime Number Theorem. It measures the baseline behavior directly.

## 2. Constraint definition

Let \(\pi(x)\) count primes less than or equal to \(x\).

The classical logarithmic density baseline is:

\[
\pi(x) \approx \frac{x}{\log x}
\]

Density drift is measured by:

\[
drift(x) =
\frac{|\pi(x) - x/\log x|}{x/\log x}
\]

A bounded CGCS score is defined as:

\[
CGCS_{density} =
\frac{1}{1 + \operatorname{mean}(drift(x))}
\]

In [ ]:
# Parameters

N_MAX = 1_000_000
N_POINTS = 80
RANDOM_SEED = 9423

params = {
    "N_MAX": N_MAX,
    "N_POINTS": N_POINTS,
    "RANDOM_SEED": RANDOM_SEED,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
    "constraint": "prime counting compared to x/log(x)",
}

params

## 3. Data generation

Generate primes below \(N_{\max}\), then evaluate \(\pi(x)\) on logarithmically spaced \(x\)-values.

In [ ]:
def sieve(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=int)
    s = np.ones(n + 1, dtype=bool)
    s[:2] = False
    for i in range(2, int(math.sqrt(n)) + 1):
        if s[i]:
            s[i*i:n+1:i] = False
    return np.nonzero(s)[0]

primes = sieve(N_MAX)

x_values = np.unique(np.logspace(2, np.log10(N_MAX), N_POINTS).astype(int))
x_values = x_values[x_values >= 10]

summary = {
    "n_max": int(N_MAX),
    "prime_count_leq_nmax": int(len(primes)),
    "x_value_count": int(len(x_values)),
    "first_primes": primes[:10].tolist(),
    "last_primes": primes[-10:].tolist(),
    "first_x_values": x_values[:10].tolist(),
    "last_x_values": x_values[-10:].tolist(),
}

summary

## 4. Measurement

For each \(x\), compute:

\[
\pi(x)
\]

\[
x/\log(x)
\]

\[
\pi(x)/(x/\log(x))
\]

\[
drift(x)
\]

In [ ]:
pi_x = np.searchsorted(primes, x_values, side="right")
baseline_xlogx = x_values / np.log(x_values)
ratio = pi_x / baseline_xlogx
drift = np.abs(pi_x - baseline_xlogx) / baseline_xlogx

density_df = pd.DataFrame({
    "x": x_values,
    "pi_x": pi_x,
    "x_over_log_x": baseline_xlogx,
    "ratio_pi_over_xlogx": ratio,
    "drift_density": drift,
})

density_df.head()

## 5. CGCS score and recoverability

The density CGCS score is:

\[
CGCS_{density} =
\frac{1}{1 + \operatorname{mean}(drift(x))}
\]

Recoverability is interpreted as global count-scale recovery:

> \(x/\log(x)\) recovers the global scale of \(\pi(x)\), not exact prime locations.

In [ ]:
mean_drift = float(density_df["drift_density"].mean())
median_drift = float(density_df["drift_density"].median())
max_drift = float(density_df["drift_density"].max())
final_ratio = float(density_df["ratio_pi_over_xlogx"].iloc[-1])
final_drift = float(density_df["drift_density"].iloc[-1])

cgcs_density = float(1.0 / (1.0 + mean_drift))

# Bin-level recovery: compare final-scale and median-scale agreement.
recoverability_score = float(1.0 / (1.0 + median_drift))

measurement = {
    "mean_drift_density": mean_drift,
    "median_drift_density": median_drift,
    "max_drift_density": max_drift,
    "final_ratio_pi_over_xlogx": final_ratio,
    "final_drift_density": final_drift,
    "cgcs_density": cgcs_density,
}

cgcs = {
    "score": cgcs_density,
    "definition": "CGCS_density = 1 / (1 + mean(|pi(x)-x/log(x)|/(x/log(x))))",
    "interpretation": "Closer to 1 indicates lower average drift from logarithmic density scale.",
}

recoverability = {
    "recoverability_score_median_density": recoverability_score,
    "definition": "1 / (1 + median density drift)",
    "note": "x/log(x) recovers global density scale, not exact prime positions",
}

measurement, recoverability

## 6. Figure 1 — prime count versus logarithmic baseline

Compare observed \(\pi(x)\) with \(x/\log(x)\).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(density_df["x"], density_df["pi_x"], marker="o", markersize=3, label="observed π(x)")
ax.plot(density_df["x"], density_df["x_over_log_x"], marker="o", markersize=3, label="x/log(x)")
ax.set_xscale("log")
ax.set_title("Prime count versus x/log(x)")
ax.set_xlabel("x")
ax.set_ylabel("count")
ax.legend()
ax.grid(True, alpha=0.3)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_pi_x_vs_x_over_log_x.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

## 7. Figure 2 — ratio to baseline

Plot:

\[
\frac{\pi(x)}{x/\log(x)}
\]

A ratio near 1 means the baseline recovers the scale of the count.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(density_df["x"], density_df["ratio_pi_over_xlogx"], marker="o", markersize=3)
ax.axhline(1.0, linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_title("Ratio: π(x) / (x/log(x))")
ax.set_xlabel("x")
ax.set_ylabel("ratio")
ax.grid(True, alpha=0.3)

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_ratio_pi_x_over_xlogx.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

fig2_path

## 8. Figure 3 — density drift

Plot:

\[
drift(x) =
\frac{|\pi(x) - x/\log x|}{x/\log x}
\]

This makes deviation from the baseline visible.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(density_df["x"], density_df["drift_density"], marker="o", markersize=3)
ax.set_xscale("log")
ax.set_title("Density drift from x/log(x)")
ax.set_xlabel("x")
ax.set_ylabel("relative drift")
ax.grid(True, alpha=0.3)

fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_density_drift.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()

fig3_path

## 9. Interpretation

1. **What continues?**  
   Prime counts continue increasing across scale.

2. **What remains under constraint?**  
   Prime counts remain close to the logarithmic density baseline.

3. **What drifts?**  
   The difference between \(\pi(x)\) and \(x/\log(x)\) is measurable drift.

4. **What is recoverable?**  
   Global density scale is recoverable from \(x/\log(x)\). Exact prime positions are not.

5. **What should not be overclaimed?**  
   This notebook measures a classical density baseline. It does not prove the Prime Number Theorem or RH.

In [ ]:
interpretation_lines = [
    f"# {NOTEBOOK_TITLE}",
    "",
    "## Constraint result",
    "",
    "This notebook measured prime density by comparing observed prime counts to the logarithmic baseline:",
    "",
    "pi(x) approximately x/log(x).",
    "",
    f"For primes up to {N_MAX:,}, the final sampled ratio was:",
    "",
    f"- final pi(x)/(x/log(x)) = {final_ratio:.6f}",
    f"- final density drift = {final_drift:.6f}",
    "",
    "## Continues",
    "",
    "Prime counts continue increasing across scale. The sequence is locally irregular but globally countable.",
    "",
    "## Remains under constraint",
    "",
    "The observed count pi(x) remains close to the logarithmic density baseline x/log(x) at large scales.",
    "",
    "## Drift",
    "",
    "Density drift was measured as |pi(x) - x/log(x)| / (x/log(x)).",
    "",
    f"- mean density drift = {mean_drift:.6f}",
    f"- median density drift = {median_drift:.6f}",
    f"- max density drift = {max_drift:.6f}",
    "",
    "## CGCS score",
    "",
    "CGCS_density = 1 / (1 + mean density drift).",
    "",
    f"- CGCS_density = {cgcs_density:.6f}",
    "",
    "## Recoverability",
    "",
    "The baseline x/log(x) recovers the global count scale of primes, not exact prime locations.",
    "",
    "## Caution",
    "",
    "This notebook does not prove the Prime Number Theorem or RH. It provides a reproducible measurement of global density structure and drift.",
]

interpretation = "\n".join(interpretation_lines)

figure_paths = [fig1_path, fig2_path, fig3_path]
figure_titles = [
    "Prime count versus x/log(x)",
    "Ratio to logarithmic baseline",
    "Density drift from x/log(x)",
]

figures_md = "\n\n## Figures\n\n"
for i, (fig, title) in enumerate(zip(figure_paths, figure_titles), start=1):
    figures_md += f"### Figure {i} — {title}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

print(interpretation + figures_md)

## 10. Export data, notes, math, and TeX

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **summary,
    **measurement,
    **recoverability,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
density_path = DATA_DIR / f"{NOTEBOOK_NUM}_density_vs_log.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
density_df.to_csv(density_path, index=False)

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "recoverability": recoverability,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "density_vs_log": str(density_path),
    },
    "docs": {
        "interpretation": str(interpretation_path),
        "design_notes": str(design_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
interpretation_path.write_text(interpretation + figures_md + "\n", encoding="utf-8")

design_lines = [
    f"# Design Notes — {NOTEBOOK_TITLE}",
    "",
    "## Notebook role",
    "",
    "Notebook 03 follows Notebook 02 by moving from local gap structure to global density structure.",
    "",
    "Notebook 01 measured finite residue structure.",
    "Notebook 02 measured local spacing through gaps.",
    "Notebook 03 measures global prime-counting density.",
    "",
    "## Constraint",
    "",
    "Prime counts are compared to the logarithmic density baseline:",
    "",
    "pi(x) approximately x/log(x).",
    "",
    "## Measurement",
    "",
    "The notebook computes:",
    "",
    "1. pi(x) for sampled x-values",
    "2. x/log(x) baseline",
    "3. ratio pi(x)/(x/log(x))",
    "4. density drift",
    "5. CGCS density score",
    "",
    "## CGCS score",
    "",
    "CGCS_density = 1 / (1 + mean(|pi(x)-x/log(x)|/(x/log(x))))",
    "",
    "## Drift",
    "",
    "drift(x) = |pi(x) - x/log(x)| / (x/log(x))",
    "",
    "## Figures",
    "",
    "1. prime count versus x/log(x)",
    "2. ratio to logarithmic baseline",
    "3. density drift from x/log(x)",
    "",
    "## Recoverability",
    "",
    "x/log(x) recovers global density scale, not exact prime locations.",
    "",
    "## Handoff",
    "",
    "Notebook 04 should measure sieve constraints as layered filtering.",
]

design_path.write_text("\n".join(design_lines) + "\n", encoding="utf-8")

summary_tex_lines = [
    rf"\section*{{{NOTEBOOK_TITLE}}}",
    "",
    r"This notebook compares prime counts to the logarithmic density baseline",
    r"\[",
    r"\pi(x) \approx \frac{x}{\log x}.",
    r"\]",
    "",
    rf"For primes up to {N_MAX:,}:",
    r"\begin{itemize}",
    rf"  \item final ratio $\pi(x)/(x/\log x) = {final_ratio:.6f}$",
    rf"  \item mean density drift $= {mean_drift:.6f}$",
    rf"  \item median density drift $= {median_drift:.6f}$",
    rf"  \item $CGCS_{{density}} = {cgcs_density:.6f}$",
    r"\end{itemize}",
    "",
    r"The baseline $x/\log x$ recovers global density scale, not exact prime locations.",
]

summary_tex_path.write_text("\n".join(summary_tex_lines) + "\n", encoding="utf-8")

math_tex_lines = [
    r"\documentclass{article}",
    r"\usepackage{amsmath}",
    r"\usepackage{amssymb}",
    r"\usepackage[margin=1in]{geometry}",
    "",
    r"\begin{document}",
    "",
    r"\section*{Math Notes: Prime Density vs x/log(x)}",
    "",
    r"\subsection*{Prime-counting function}",
    "",
    r"Let $\pi(x)$ denote the number of primes less than or equal to $x$.",
    "",
    r"\subsection*{Logarithmic density baseline}",
    "",
    r"\[",
    r"\pi(x) \approx \frac{x}{\log x}.",
    r"\]",
    "",
    r"\subsection*{Ratio}",
    "",
    r"\[",
    r"R(x) = \frac{\pi(x)}{x/\log x}.",
    r"\]",
    "",
    r"\subsection*{Density drift}",
    "",
    r"\[",
    r"drift(x) =",
    r"\frac{|\pi(x) - x/\log x|}{x/\log x}.",
    r"\]",
    "",
    r"\subsection*{CGCS score}",
    "",
    r"\[",
    r"CGCS_{density} =",
    r"\frac{1}{1 + \operatorname{mean}(drift(x))}.",
    r"\]",
    "",
    r"Measured value:",
    r"\[",
    rf"CGCS_{{density}} = {cgcs_density:.6f}.",
    r"\]",
    "",
    r"\subsection*{Recoverability}",
    "",
    r"The baseline $x/\log x$ recovers global count scale, not exact prime locations.",
    "",
    r"\end{document}",
]

math_tex_path.write_text("\n".join(math_tex_lines) + "\n", encoding="utf-8")

summary_path, density_path, metadata_path, interpretation_path, design_path, summary_tex_path, math_tex_path

## 11. Export zip

Pi-stage-lab style root export zip, with optional Colab download lines left commented.

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 12. Next notebook handoff

Next notebook:

```text
04_sieve_constraints.ipynb
```

Purpose:

> measure sieve constraints as layered filtering, showing how candidate sets remain under successive divisibility constraints.

In [ ]:
next_step = "Notebook 04: sieve constraints as layered filtering."
print(next_step)